In [1]:
import logging

import pandas as pd
import torch
import torchtext

from models import *
from preprocessing import *

print("PyTorch Version : {}".format(torch.__version__))
print("Torch Text Version : {}".format(torchtext.__version__))

/home/sigma/projects/PycharmProjects/toxic-comment-detection/venv/lib/python3.11/site-packages/torchtext/data/utils.py:105: UserWarning: Spacy model "en" could not be loaded, trying "en_core_web_sm" instead
  warnings.warn(


PyTorch Version : 2.0.1+cu118
Torch Text Version : 0.15.2+cpu


In [2]:
%env WANDB_SILENT=True


logger = logging.getLogger("wandb")
logger.setLevel(logging.ERROR)

env: WANDB_SILENT=True


In [3]:
device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda:0")

device

device(type='cuda', index=0)

In [4]:
# Load Datasets And Create Data Loaders

train_df = pd.read_csv("data/train.csv")
train_df.describe()

,toxic,severe_toxic,obscene,threat,insult,identity_hate
count,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000
mean,0.095844,0.009996,0.052948,0.002996,0.049364,0.008805
std,0.294379,0.099477,0.223931,0.054650,0.216627,0.093420
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [5]:
BATCH_SIZE = 256
# df_frac = 0.3
df_frac = 1.0
num_epochs = 50

## CNN-BiGRU (50 tokens - GlovE 300 - adadelta)

In [6]:
train_dl_50t_glove_300, val_dl_50t_glove_300 = get_dataloaders(
    target_convertor="glove_300",
    maximum_tokens=50,
    batch_size=BATCH_SIZE,
    dataset_fraction=df_frac,
)

127656it [01:14, 1723.72it/s]
31915it [00:18, 1741.81it/s]


In [7]:
cnn_bigru = CNNBiGRU(
    embed_dim=300,
    conv_config={"num_channels": 50, "kernel_sizes": [1, 2, 3, 5]},
    optimizer_type="adadelta",
    learning_rate=1.0,
).to(device)

cnn_bigru.train_model(
    train_dataloader=train_dl_50t_glove_300,
    validation_dataloader=val_dl_50t_glove_300,
    num_of_epochs=num_epochs,
    device=device,
    name="CNN-BiGRU (50 tokens - Glove_300 - adadelta)",
    architecture="CNN-BiGRU",
)

Epoch 1:
{   'Training AUROC': 0.956955075263977,
    'Training F1_score': 0.6028831601142883,
    'Training accuracy': 0.9759392142295837,
    'Training loss': 0.07474114336745295,
    'Training precision': 0.7599413394927979,
    'Training recall': 0.49962499737739563,
    'Validation AUROC': 0.977846622467041,
    'Validation F1_score': 0.6856145858764648,
    'Validation accuracy': 0.9802443981170654,
    'Validation loss': 0.058109135538339615,
    'Validation precision': 0.8360356688499451,
    'Validation recall': 0.5810677409172058}
Epoch 2:
{   'Training AUROC': 0.9806759357452393,
    'Training F1_score': 0.7169909477233887,
    'Training accuracy': 0.9815219640731812,
    'Training loss': 0.052263494648173724,
    'Training precision': 0.814538836479187,
    'Training recall': 0.6403085589408875,
    'Validation AUROC': 0.9807753562927246,
    'Validation F1_score': 0.7099741697311401,
    'Validation accuracy': 0.9812313914299011,
    'Validation loss': 0.05363363112509251,

## CNN-BiGRU (25 tokens - GlovE 300 - adadelta)

In [6]:
train_dl_25t_glove_300, val_dl_25t_glove_300 = get_dataloaders(
    target_convertor="glove_300",
    maximum_tokens=25,
    batch_size=BATCH_SIZE,
    dataset_fraction=df_frac,
)

127656it [01:04, 1987.73it/s]
31915it [00:18, 1758.24it/s]


In [7]:
cnn_bigru = CNNBiGRU(
    embed_dim=300,
    conv_config={"num_channels": 50, "kernel_sizes": [1, 2, 3, 5]},
    optimizer_type="adadelta",
    learning_rate=1.0,
).to(device)

cnn_bigru.train_model(
    train_dataloader=train_dl_25t_glove_300,
    validation_dataloader=val_dl_25t_glove_300,
    num_of_epochs=num_epochs,
    device=device,
    name="CNN-BiGRU (25 tokens - Glove_300 - adadelta)",
    architecture="CNN-BiGRU",
)

Epoch 1:
{   'Training AUROC': 0.9476503729820251,
    'Training F1_score': 0.5759425759315491,
    'Training accuracy': 0.9742432832717896,
    'Training loss': 0.08030181468458834,
    'Training precision': 0.7350084781646729,
    'Training recall': 0.4734758734703064,
    'Validation AUROC': 0.9689871072769165,
    'Validation F1_score': 0.6351401209831238,
    'Validation accuracy': 0.9793305397033691,
    'Validation loss': 0.06405040882527828,
    'Validation precision': 0.8516687154769897,
    'Validation recall': 0.5063942670822144}
Epoch 2:
{   'Training AUROC': 0.9704477787017822,
    'Training F1_score': 0.6875901222229004,
    'Training accuracy': 0.979921281337738,
    'Training loss': 0.06031240787887143,
    'Training precision': 0.8085228204727173,
    'Training recall': 0.5981268882751465,
    'Validation AUROC': 0.9718832969665527,
    'Validation F1_score': 0.6576936841011047,
    'Validation accuracy': 0.9803436398506165,
    'Validation loss': 0.060212832257151606,

## CNN-BiGRU (50 tokens - GlovE 100 - adadelta)

In [6]:
train_dl_50t_glove_100, val_dl_50t_glove_100 = get_dataloaders(
    target_convertor="glove_100",
    maximum_tokens=50,
    batch_size=BATCH_SIZE,
    dataset_fraction=df_frac,
)

127656it [01:11, 1792.31it/s]
31915it [00:15, 2024.39it/s]


In [7]:
cnn_bigru = CNNBiGRU(
    embed_dim=100,
    conv_config={"num_channels": 50, "kernel_sizes": [1, 2, 3, 5]},
    optimizer_type="adadelta",
    learning_rate=1.0,
).to(device)

cnn_bigru.train_model(
    train_dataloader=train_dl_50t_glove_100,
    validation_dataloader=val_dl_50t_glove_100,
    num_of_epochs=num_epochs,
    device=device,
    name="CNN-BiGRU (50 tokens - Glove_100 - adadelta)",
    architecture="CNN-BiGRU",
)

Epoch 1:
{   'Training AUROC': 0.9229296445846558,
    'Training F1_score': 0.4319147765636444,
    'Training accuracy': 0.9703369736671448,
    'Training loss': 0.09433420360058486,
    'Training precision': 0.7186121940612793,
    'Training recall': 0.30873996019363403,
    'Validation AUROC': 0.9557133913040161,
    'Validation F1_score': 0.4999016225337982,
    'Validation accuracy': 0.9734503030776978,
    'Validation loss': 0.07933291929960251,
    'Validation precision': 0.8350312113761902,
    'Validation recall': 0.35673171281814575}
Epoch 2:
{   'Training AUROC': 0.9566895961761475,
    'Training F1_score': 0.5739683508872986,
    'Training accuracy': 0.9751585125923157,
    'Training loss': 0.07365734571729132,
    'Training precision': 0.7681289911270142,
    'Training recall': 0.45815905928611755,
    'Validation AUROC': 0.9622097015380859,
    'Validation F1_score': 0.5898577570915222,
    'Validation accuracy': 0.9754608869552612,
    'Validation loss': 0.071577893644571

## CNN-BiGRU (25 tokens - GlovE 100 - adadelta)

In [6]:
train_dl_25t_glove_100, val_dl_25t_glove_100 = get_dataloaders(
    target_convertor="glove_100",
    maximum_tokens=25,
    batch_size=BATCH_SIZE,
    dataset_fraction=df_frac,
)

127656it [01:03, 2000.15it/s]
31915it [00:17, 1776.17it/s]


In [7]:
cnn_bigru = CNNBiGRU(
    embed_dim=100,
    conv_config={"num_channels": 50, "kernel_sizes": [1, 2, 3, 5]},
    optimizer_type="adadelta",
    learning_rate=1.0,
).to(device)

cnn_bigru.train_model(
    train_dataloader=train_dl_25t_glove_100,
    validation_dataloader=val_dl_25t_glove_100,
    num_of_epochs=num_epochs,
    device=device,
    name="CNN-BiGRU (25 tokens - Glove_100 - adadelta)",
    architecture="CNN-BiGRU",
)

Epoch 1:
{   'Training AUROC': 0.9120911955833435,
    'Training F1_score': 0.4083987772464752,
    'Training accuracy': 0.9692102074623108,
    'Training loss': 0.09932869548847775,
    'Training precision': 0.6997936964035034,
    'Training recall': 0.28833553194999695,
    'Validation AUROC': 0.9449253678321838,
    'Validation F1_score': 0.5603547692298889,
    'Validation accuracy': 0.9741135239601135,
    'Validation loss': 0.07957738029956818,
    'Validation precision': 0.7166515588760376,
    'Validation recall': 0.46002620458602905}
Epoch 2:
{   'Training AUROC': 0.9468486309051514,
    'Training F1_score': 0.5465174913406372,
    'Training accuracy': 0.9740735292434692,
    'Training loss': 0.07904520524079671,
    'Training precision': 0.7690725922584534,
    'Training recall': 0.4238602817058563,
    'Validation AUROC': 0.9524575471878052,
    'Validation F1_score': 0.5647546648979187,
    'Validation accuracy': 0.9758681654930115,
    'Validation loss': 0.0740580393970012